# FFNN 振幅头 + PennyLane PQC 相位头 混合式 VMC

FFNN 振幅头 + PennyLane PQC 相位头 混合式 VMC
================================================
系统：H2 / 6-31G（SINGLE_SIZE=4 自旋轨道，物理空间 16 维）
基准：FCI E0 = -1.05434745 Ha

结构（对应 FFNN振幅相位分离版-VMC-3.ipynb 的改造点）：
    σ(4) → FFNN backbone(tanh ×2) → h(hidden)
    logA = Linear(h → 1)                         # 经典振幅头（读 full hidden）
    z    = tanh(Linear(h → 4))                   # 相位支路压缩特征
    φ    = W · PQC(z) + b                        # 相位头 = PennyLane 参数化量子线路
    logψ = logA + i·φ                            # 实参数、复输出

PQC（4 qubits, L 层）：
    编码层  RX(π·z_i)
    变分层  CNOT 链 (0-1,1-2,2-3) + RY(a)/RZ(b) 每比特
    特征    f_i = <Z_i>（4 维实数）→ 经典线性读出 φ
    梯度    diff_method="backprop"（default.qubit 态矢量反传，jax.grad 直通）

VMC 引擎沿用已验证框架（0925 失败/成功分析文档）：
    - 实参数 Wirtinger 梯度 ∇logψ = ∇Re + i·∇Im
    - machine_pow=1 按 |ψ| 采样 + 重要性加权 w ∝ |ψ|（防零方差冻结）
    - 自然梯度 SR：∇E = 2Re⟨(E_loc−⟨E⟩)(∇logψ)*⟩，度量 Re(S)

用法：
    python FFNN振幅_PQC相位混合版-VMC.py [--n-iter 400] [--seed 21] [--lr 0.1]
        [--layers 2] [--hidden 12] [--pqc-scale 0.1] [--diag-shift 0.01]

In [1]:
import argparse
import os
import time

import jax
import jax.numpy as jnp
from jax import flatten_util
from functools import partial
import flax.nnx as nnx
import netket as nk
import numpy as np
import optax
import pennylane as qml

jax.config.update("jax_enable_x64", True)

import sys
try:
    _HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # notebook 环境无 __file__
    _HERE = os.getcwd()
    if not os.path.exists(os.path.join(_HERE, "H2_6_31G.py")):
        _HERE = "/Users/yangjianfei/mac_vscode/神经网络量子态/NES_VMC/experiments/H2分子/量子-经典混合式 Ansatz"
sys.path.append(os.path.join(_HERE, ".."))
from H2_6_31G import (
    SINGLE_SIZE, ha, hi, E_fcis, hf_ground_energy, single_rule,
)

/opt/miniconda3/envs/Netket/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: You can load logged data with nk.utils.history.HistoryDict.from_file(data.log).

H2 分子基本信息
HF energy = -0.99749729 Ha
Total electrons = (1, 1)
Total basis functions = 4
H₂ FCI 基准能量
E0 = -1.05434745 Ha  |  激发能: 0.0000 eV
E1 = -0.95790573 Ha  |  激发能: 2.6243 eV
E2 = -0.66895227 Ha  |  激发能: 10.4871 eV
E3 = -0.55140192 Ha  |  激发能: 13.6859 eV
E4 = -0.02803278 Ha  |  激发能: 27.9275 eV
E5 = -0.01450343 Ha  |  激发能: 28.2956 eV
E6 = -0.00518297 Ha  |  激发能: 28.5492 eV
E7 = 0.02397075 Ha  |  激发能: 29.3425 eV
E8 = 0.23432390 Ha  |  激发能: 35.0666 eV
E9 = 0.25720575 Ha  |  激发能: 35.6892 eV
E10 = 0.49958128 Ha  |  激发能: 42.2846 eV
E11 = 0.50024484 Ha  |  激发能: 42.3026 eV

HF reference state: [0 0 0 1 0 0 0 1]
single_edges: [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3), (4, 5), (4, 6), (4, 7), (5, 6), (5, 7), (6, 7)]


## 1. PennyLane PQC（4 qubits）

In [2]:
# ============================================================
# 1. PennyLane PQC（模块级 QNode，避免 nnx.split 携带设备对象）
# ============================================================
N_QUBITS = 4  # PQC 编码 4 维经典瓶颈特征 z ∈ [-1,1]^4


def pqc_circuit(h, weights, n_qubits=N_QUBITS, n_layers=2):
    """批量 PQC。

    h:       (B, n_qubits) 实数特征（经典网络输出，已 tanh 到 [-1,1]）
    weights: (n_layers, 2*n_qubits) 变分参数
    返回：n_qubits 个 <Z_i>，每个形状 (B,)
    """
    for i in range(n_qubits):
        qml.RX(h[:, i] * np.pi, wires=i)
    qml.Barrier(wires=range(n_qubits))

    for layer in range(n_layers):
        for i in range(n_qubits - 1):
            qml.CNOT(wires=[i, i + 1])
        qml.Barrier(wires=range(n_qubits))
        for i in range(n_qubits):
            qml.RY(weights[layer, 2 * i], wires=i)
            qml.RZ(weights[layer, 2 * i + 1], wires=i)

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]


_DEV = qml.device("default.qubit", wires=N_QUBITS)
PQC_NODE = qml.QNode(
    pqc_circuit, _DEV, interface="jax", diff_method="backprop"
)

## 2. Ansatz：FFNN 振幅头 + PQC 相位头

In [3]:
# ============================================================
# 2. Ansatz：FFNN 振幅头 + PQC 相位头
# ============================================================
class SingleStateAnsatz_FFNN_PQC(nnx.Module):
    def __init__(
        self,
        n_spin_orbitals: int,
        hidden_dim: int = 12,
        qubits: int = N_QUBITS,
        n_layers: int = 2,
        pqc_scale: float = 0.1,
        *,
        rngs: nnx.Rngs,
    ):
        super().__init__()
        self.qubits = qubits
        self.n_layers = n_layers
        self.hidden_dim = hidden_dim

        # 经典骨干（实参数）
        self.backbone1 = nnx.Linear(n_spin_orbitals, hidden_dim, rngs=rngs, param_dtype=float)
        self.backbone2 = nnx.Linear(hidden_dim, hidden_dim, rngs=rngs, param_dtype=float)
        # 相位支路压缩层：hidden -> qubits（PQC 编码输入）
        self.phase_proj = nnx.Linear(hidden_dim, qubits, rngs=rngs, param_dtype=float)

        # 振幅头（经典，直接读 full hidden，保证振幅表达力）
        self.head_logA = nnx.Linear(hidden_dim, 1, rngs=rngs, param_dtype=float)

        # 相位头 = PQC 变分参数 + 线性读出
        key = rngs.params()
        self.qnn_params = nnx.Param(
            jax.random.uniform(key, (n_layers, 2 * qubits), dtype=jnp.float64)
            * pqc_scale
            * np.pi
        )
        self.readout = nnx.Linear(qubits, 1, rngs=rngs, param_dtype=float)

    def __call__(self, x: jax.Array) -> jax.Array:
        h = nnx.tanh(self.backbone1(x))
        h = nnx.tanh(self.backbone2(h))

        # 振幅头：经典，读 full hidden
        logA = jnp.squeeze(self.head_logA(h), axis=-1)

        # 相位头：PQC（4 qubits），输入是 tanh 限幅的 4 维压缩特征
        z = nnx.tanh(self.phase_proj(h))          # (..., qubits)
        lead_shape = jnp.shape(z)[:-1]
        z2 = jnp.reshape(z, (-1, self.qubits))
        feats = PQC_NODE(z2, self.qnn_params, n_layers=self.n_layers)
        feats = jnp.stack(feats, axis=1)          # (B, qubits)
        phi = jnp.squeeze(self.readout(feats), axis=-1)
        phi = jnp.reshape(phi, lead_shape)

        return logA + 1j * phi

## 3. VMC 引擎（实参数 Wirtinger 梯度 + 重要性加权 + 自然梯度 SR）

In [4]:
# ============================================================
# 3. VMC 引擎（沿用已验证实现）
# ============================================================
def create_machine(model: nnx.Module):
    graphdef, state = nnx.split(model)

    @jax.jit
    def machine(params, sigma):
        m = nnx.merge(graphdef, params)
        return m(sigma)

    return machine, graphdef, state


@partial(jax.jit, static_argnames=("machine",))
def compute_local_energies(machine, params, sigma):
    eta, H_eta = ha.get_conn_padded(sigma)
    logpsi_sigma = machine(params, sigma)
    # eta 为 3D (N, n_conn, size)，PQC 只支持单一批维 → 展平后计算再还原
    n_conn = eta.shape[-2]
    logpsi_eta = machine(params, eta.reshape(-1, sigma.shape[-1]))
    logpsi_eta = logpsi_eta.reshape(logpsi_sigma.shape + (n_conn,))
    return jnp.sum(H_eta * jnp.exp(logpsi_eta - logpsi_sigma[..., None]), axis=-1)


def _grad_logpsi_real_params(machine, params, sigma):
    """实参数、复输出：∇logψ = ∇Re + i·∇Im（手写 Wirtinger，绕开 holomorphic 限制）。"""
    re_grad = jax.vmap(lambda s: jax.grad(lambda p: machine(p, s).real)(params))(sigma)
    im_grad = jax.vmap(lambda s: jax.grad(lambda p: machine(p, s).imag)(params))(sigma)
    return jax.tree_util.tree_map(lambda r, i: r + 1j * i, re_grad, im_grad)


@partial(jax.jit, static_argnames=("machine",))
def _sample_weights(machine, params, sigma):
    """w_i = |ψ(σ_i)|，配合 p ∝ |ψ| 采样（machine_pow=1）的重要性权重。"""
    logpsi = machine(params, sigma)
    return jnp.exp(jnp.real(logpsi))


def _weighted_mean_std(x, w):
    w_sum = jnp.sum(w)
    mean = jnp.sum(w * x) / w_sum
    var = jnp.sum(w * jnp.abs(x - mean) ** 2) / w_sum
    return mean, jnp.sqrt(var / x.shape[0])


@partial(jax.jit, static_argnames=("machine",))
def forces_expect_hermitian(machine, params, sigma):
    """重要性加权 force 梯度（实参数，取实部）：∇⟨E⟩ = 2·Re⟨(E_loc−⟨E⟩)(∇logψ)*⟩_w。"""
    w = _sample_weights(machine, params, sigma)
    O_loc = compute_local_energies(machine, params, sigma)
    O_mean, O_std = _weighted_mean_std(O_loc, w)
    O_centered = O_loc - O_mean

    grad_matrix = _grad_logpsi_real_params(machine, params, sigma)

    def weight_and_mean(grad_component):
        wc = (w * O_centered).reshape((w.shape[0],) + (1,) * (grad_component.ndim - 1))
        return (jnp.sum(wc * jnp.conj(grad_component), axis=0) / jnp.sum(w)).real

    grad = jax.tree_util.tree_map(weight_and_mean, grad_matrix)
    return O_mean, O_std, grad


def compute_qgt(machine, params, sigma, diag_shift=0.01):
    """加权 QGT（实参数，度量取 Re(S)）。"""
    n_samples = sigma.shape[0]
    w = _sample_weights(machine, params, sigma)
    grad_matrix = _grad_logpsi_real_params(machine, params, sigma)
    grad_flat, unravel_fn = flatten_util.ravel_pytree(grad_matrix)
    grad_flat = grad_flat.reshape(n_samples, -1)

    w_mean = jnp.sum(w[:, None] * grad_flat, axis=0, keepdims=True) / jnp.sum(w)
    grad_centered = grad_flat - w_mean
    qw = (w[:, None] * jnp.conj(grad_centered)).T @ grad_centered / jnp.sum(w)
    qgt = qw.real
    qgt_reg = qgt + diag_shift * jnp.eye(qgt.shape[0])
    return qgt_reg, unravel_fn


_EXACT_STATES_INT = jnp.asarray(np.array(hi.all_states()), dtype=jnp.int32)
_EXACT_STATES = _EXACT_STATES_INT.astype(jnp.float64)
_EXACT_ETA, _EXACT_HE = ha.get_conn_padded(_EXACT_STATES_INT)


@partial(jax.jit, static_argnames=("machine",))
def exact_energy(machine, params):
    """全空间 16 组态精确能量 E = Σ p_σ E_loc(σ)，消除采样噪声的收敛判据。"""
    logpsi = machine(params, _EXACT_STATES)
    n_conn = _EXACT_ETA.shape[-2]
    logpsi_eta = machine(
        params, _EXACT_ETA.reshape(-1, _EXACT_STATES.shape[-1])
    ).reshape(logpsi.shape[0], n_conn)
    e_loc = jnp.sum(_EXACT_HE * jnp.exp(logpsi_eta - logpsi[:, None]), axis=-1)
    psi = jnp.exp(logpsi)
    prob = jnp.abs(psi) ** 2
    prob = prob / jnp.sum(prob)
    return jnp.sum(prob * e_loc).real

## 4. 训练函数

In [5]:
# ============================================================
# 4. 训练
# ============================================================
def run(n_iter=400, seed=21, lr=0.1, layers=2, hidden=12, pqc_scale=0.1,
        diag_shift=0.01, n_chains=150, sweep_size=32, chain_length=20,
        log_every=50, tag=""):
    model = SingleStateAnsatz_FFNN_PQC(
        n_spin_orbitals=SINGLE_SIZE, hidden_dim=hidden, n_layers=layers,
        pqc_scale=pqc_scale, rngs=nnx.Rngs(seed),
    )
    sampler = nk.sampler.MetropolisSampler(
        hi, rule=single_rule, n_chains=n_chains, sweep_size=sweep_size, machine_pow=1
    )
    machine, graphdef, params = create_machine(model)
    sampler_state = sampler.init_state(machine, params, seed=1)

    optimizer = optax.sgd(learning_rate=lr)
    opt_state = optimizer.init(params)

    history = {"step": [], "energy": [], "energy_std": [], "error": [], "exact": [], "exact_err": []}

    print("\n" + "=" * 64)
    print(f"开始 FFNN振幅+PQC相位 混合 VMC | layers={layers} hidden={hidden} "
          f"pqc_scale={pqc_scale} lr={lr} diag_shift={diag_shift} seed={seed} {tag}")
    print("=" * 64)

    samples = None
    t0 = time.time()
    for step in range(n_iter):
        sampler_state = sampler.reset(machine, params, sampler_state)
        samples, sampler_state = sampler.sample(
            machine, params, state=sampler_state, chain_length=chain_length
        )
        samples = samples.reshape(-1, hi.size)

        energy, energy_std, grad = forces_expect_hermitian(machine, params, samples)

        qgt_reg, _ = compute_qgt(machine, params, samples, diag_shift=diag_shift)
        grad_flat, grad_unravel_fn = flatten_util.ravel_pytree(grad)
        natural_grad = jnp.linalg.solve(qgt_reg, grad_flat)
        grad = grad_unravel_fn(natural_grad)

        updates, opt_state = optimizer.update(grad, opt_state, params)
        params = optax.apply_updates(params, updates)

        if step % log_every == 0 or step == n_iter - 1:
            ex = float(exact_energy(machine, params))
            error = abs(float(energy.real) - float(E_fcis[0]))
            ex_err = abs(ex - float(E_fcis[0])) * 1000.0  # mHa
            history["step"].append(step)
            history["energy"].append(float(energy.real))
            history["energy_std"].append(float(energy_std))
            history["error"].append(error)
            history["exact"].append(ex)
            history["exact_err"].append(ex_err)
            print(f"Step {step:3d} | E_sample: {energy.real:.8f} ± {energy_std:.6f} "
                  f"| E_exact: {ex:.8f} | 精确误差: {ex_err:.3f} mHa")

    final_energy, final_std, _ = forces_expect_hermitian(machine, params, samples)
    final_exact = float(exact_energy(machine, params))
    final_err_mHa = abs(final_exact - float(E_fcis[0])) * 1000.0
    print("\n" + "=" * 64)
    print(f"训练完成! 用时 {time.time() - t0:.1f}s")
    print(f"采样能量：{final_energy.real:.8f} ± {final_std:.6f} Ha")
    print(f"全空间精确能量：{final_exact:.8f} Ha")
    print(f"FCI 基准：{E_fcis[0]:.8f} Ha")
    print(f"绝对误差：{abs(final_err_mHa):.4f} mHa  |  相对误差：{abs(final_err_mHa)/abs(float(E_fcis[0]))/1000*100:.4f}%")
    print("=" * 64)

    return model, machine, params, history, final_exact, final_err_mHa

## 5. 运行 400 步 + 绘图

In [7]:
model, machine, params, history, final_exact, final_err_mHa = run(
    n_iter=400, seed=21, lr=0.1, layers=2, hidden=12,
    pqc_scale=0.1, diag_shift=0.01, tag='',
)

print(f"最终精确能量 = {final_exact:.8f} Ha, 误差 = {final_err_mHa:.3f} mHa")

import matplotlib
#matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.errorbar(history["step"], history["energy"], yerr=history["energy_std"],
            fmt="o-", capsize=3, label="VMC energy (sampled)")
ax.plot(history["step"], history["exact"], "s--", ms=4,
        label="VMC energy (exact sum)")
ax.axhline(float(E_fcis[0]), color="r", ls="--",
           label=f"FCI E0 = {float(E_fcis[0]):.6f} Ha")
ax.axhline(float(hf_ground_energy), color="g", ls=":",
           label=f"HF = {float(hf_ground_energy):.6f} Ha")
ax.set_xlabel("iteration")
ax.set_ylabel("Energy (Ha)")
ax.set_ylim(-1.06, -0.94)
ax.set_title("FFNN amplitude + PQC phase hybrid VMC (H2/6-31G)")
ax.legend()
fig.tight_layout()
out_png = "FFNN振幅_PQC相位混合版-VMC.png"
fig.savefig(out_png, dpi=150)
print(f"训练曲线已保存: {out_png}")


开始 FFNN振幅+PQC相位 混合 VMC | layers=2 hidden=12 pqc_scale=0.1 lr=0.1 diag_shift=0.01 seed=21 
Step   0 | E_sample: 0.85706878 ± 0.011648 | E_exact: 0.15766181 | 精确误差: 1212.009 mHa
Step  50 | E_sample: -1.03548401 ± 0.002026 | E_exact: -1.03551806 | 精确误差: 18.829 mHa
Step 100 | E_sample: -1.04584981 ± 0.001516 | E_exact: -1.04588188 | 精确误差: 8.466 mHa
Step 150 | E_sample: -1.04931323 ± 0.001334 | E_exact: -1.04914329 | 精确误差: 5.204 mHa
Step 200 | E_sample: -1.05072951 ± 0.000863 | E_exact: -1.05069068 | 精确误差: 3.657 mHa
Step 250 | E_sample: -1.05127496 ± 0.001054 | E_exact: -1.05134604 | 精确误差: 3.001 mHa
Step 300 | E_sample: -1.05167491 ± 0.001094 | E_exact: -1.05176028 | 精确误差: 2.587 mHa
Step 350 | E_sample: -1.05202431 ± 0.000927 | E_exact: -1.05199089 | 精确误差: 2.357 mHa
Step 399 | E_sample: -1.05226164 ± 0.001154 | E_exact: -1.05237680 | 精确误差: 1.971 mHa

训练完成! 用时 69.8s
采样能量：-1.05229765 ± 0.001142 Ha
全空间精确能量：-1.05237680 Ha
FCI 基准：-1.05434745 Ha
绝对误差：1.9706 mHa  |  相对误差：0.1869%
最终精确能量 = -1.05237